In [19]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGA</cre>ATTACGG<bc/>AGATCGGA')\
                  .named('template_pool')

mutated_pool = template_pool.stylize(region='cre', style='goldenrod')\
                            .mutagenize(region='cre',
                                        mutation_rate=0.1, 
                                        style='yellow bold',
                                        mode='random', 
                                        num_states=5, 
                                        prefix='mutagenize').named('mutated_pool')\

deletion_pool = template_pool.stylize(region='cre', style='salmon')\
                             .deletion_scan(region='cre', 
                                            deletion_length=6, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            style='red bold',
                                            prefix='delscan').named('deletion_pool')\
                            .repeat_states(2, prefix='v', iter_order=-2)

sites_pool=pp.from_seqs(['AAAAAA','TTTTTT'], 
                        mode='sequential', 
                        iter_order=-1,
                        prefix='site').named('sites_pool')

insertion_pool = template_pool.stylize(region='cre', style='blue')\
                              .insertion_scan(region='cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential',
                                              prefix='insscan',
                                              prefix_position='pos', 
                                              prefix_insert='ins',
                                              style='cyan bold').named('insertion_pool')
                              
shuffle_pool = template_pool.stylize(region='cre', style='purple')\
                            .shuffle_scan(region='cre', 
                                          shuffle_length=6, 
                                          shuffles_per_position=2,
                                          positions=slice(None, None, 5), 
                                          mode='sequential', 
                                          style='magenta bold',
                                          prefix='shufscan',
                                          prefix_position='pos',
                                          prefix_shuffle='shuf')
                            
L = len('GGAAAGCGGGCAGTGAGCACACAGGA')
#source_pool_a = pp.from_seq('A'*L).stylize(style='green')
#source_pool_b = pp.from_seq('T'*L).stylize(style='red')
recomb_pool = template_pool.recombine(region='cre', 
                                      num_breakpoints=2,
                                      source_pools=['A'*L, 'T'*L],
                                      styles=['green','red','green'],
                                      mode='random',
                                      num_states=5,
                                      prefix='recomb').named('recomb_pool')

combo_pool = pp.stack([mutated_pool, deletion_pool, insertion_pool, shuffle_pool, recomb_pool])\
    .named('stack_pool')\
    .insert_kmers(region='bc', mode='random', length=5, prefix='bc', style='green bold')\
    .named('combo_pool')\
    .stylize(which='tags', style='gray')

combo_pool.print_library(show_name=True,seed=12)

pool[20]: seq_length=None, num_states=40
state  name                           seq
    0  mutagenize_0.bc_0              TCCCGACT<cre>GAGAAGCGGGCAGTGAGCACACAGGA</cre>ATTACGG<bc>ACCTG</bc>AGATCGGA
    1  mutagenize_1.bc_1              TCCCGACT<cre>GGAAAGCGGGTAGTGAGCACACATGA</cre>ATTACGG<bc>CGCGC</bc>AGATCGGA
    2  mutagenize_2.bc_2              TCCCGACT<cre>GGAAAGCGGTCAGTAAGCACATAGGA</cre>ATTACGG<bc>CACTG</bc>AGATCGGA
    3  mutagenize_3.bc_3              TCCCGACT<cre>GGAAAGCGGGCAGTTAGCATACATGA</cre>ATTACGG<bc>GGGGC</bc>AGATCGGA
    4  mutagenize_4.bc_4              TCCCGACT<cre>GGAACGCGGGCAGTGAGCACACAGGA</cre>ATTACGG<bc>CGTGC</bc>AGATCGGA
    5  delscan_0.v_0.bc_5             TCCCGACT<cre>------CGGGCAGTGAGCACACAGGA</cre>ATTACGG<bc>CATCT</bc>AGATCGGA
    6  delscan_0.v_1.bc_6             TCCCGACT<cre>------CGGGCAGTGAGCACACAGGA</cre>ATTACGG<bc>AAAGT</bc>AGATCGGA
    7  delscan_1.v_0.bc_7             TCCCGACT<cre>GGAAA------AGTGAGCACACAGGA</cre>ATTACGG<bc>GGACC</bc>AGATCGGA
    8  delsca

Pool(id=20, name='pool[20]', op='op[20]:stylize', num_states=40)

In [8]:
combo_pool.print_dag()

pool[17] (pool, n=40)
└── op[17]:stylize [mode=fixed, n=1]
    └── combo_pool (pool, n=40)
        └── op[16]:get_kmers [mode=random, n=40]
            └── stack_pool (pool, n=40)
                └── op[15]:stack [mode=sequential, n=4]
                    ├── mutated_pool (pool, n=10)
                    │   └── op[2]:mutagenize [mode=random, n=10]
                    │       └── pool[1] (pool, n=1)
                    │           └── op[1]:stylize [mode=fixed, n=1]
                    │               └── template_pool (pool, n=1)
                    │                   └── op[0]:from_seq [mode=fixed, n=1]
                    ├── pool[7] (pool, n=10)
                    │   └── op[7]:repeat [mode=sequential, n=2]
                    │       └── deletion_pool (pool, n=5)
                    │           └── op[6]:deletion_scan(replace_region) [mode=fixed, n=1]
                    │               ├── pool[4] (pool, n=5)
                    │               │   └── op[4]:deletion_scan(regio

Pool(id=17, name='pool[17]', op='op[17]:stylize', num_states=40)

In [9]:
combo_pool.mutagenize?

Signature:
combo_pool.mutagenize(
    region: Union[str, collections.abc.Sequence[numbers.Integral], NoneType] = None,
    num_mutations: Optional[numbers.Integral] = None,
    mutation_rate: Optional[numbers.Real] = None,
    allowed_chars: Optional[str] = None,
    style: Optional[str] = None,
    prefix: Optional[str] = None,
    mode: Literal['random', 'sequential', 'fixed'] = 'random',
    num_states: Optional[int] = None,
    iter_order: Optional[numbers.Real] = None,
) -> 'poolparty.pool.Pool'
Docstring:
Create a Pool that applies mutations to a sequence.

Parameters
----------
region : Union[str, Sequence[Integral], None], default=None
    Region to mutagenize. Can be a marker name (str), explicit interval [start, stop],
    or None to mutagenize entire sequence. Positions are region-relative.
num_mutations : Optional[Integral], default=None
    Fixed number of mutations to apply (mutually exclusive with mutation_rate).
mutation_rate : Optional[Real], default=None
    Probabili